# 高级类型提示与数据结构

## @dataclass装饰器
在 Python 开发中，我们经常需要定义纯粹用于存储数据的类（即“数据载体”）。传统做法是编写一个包含 `__init__`、`__repr__`、`__eq__ `等样板代码（Boilerplate Code）的普通类，这不仅繁琐，而且容易出错。Python 3.7 引入的 @dataclass 装饰器能够自动为我们生成这些特殊方法，极大地减少了代码量，同时保持了代码的高可读性和类型安全性。当你在类上使用它时，Python 会根据类中定义的带有类型提示（Type Hints）的字段，自动为你生成常用特殊方法

In [4]:
from dataclasses import dataclass

@dataclass
class User:
    user_id: int
    username: str
    is_active: bool

# 实例化
user1= User(user_id=1, username='Alice', is_active=True)
print(user1)  # 输出: User(user_id=1, username='Alice', is_active=True)
user2 = User(user_id=1, username='Alice', is_active=True)
print(user1 ==user2)  

User(user_id=1, username='Alice', is_active=True)
True


## try except
在捕获常见异常（如除以零错误）时，直接将异常对象转换为字符串并进行处理。
except ZeroDivisionError as e
str(e)
当需要排查复杂函数调用中的深层错误时，格式化完整的堆栈信息为字符串。
error_str = traceback.format_exc()将当前异常的完整堆栈转为字符串

## field(default_factory=list)
在 Python 的数据类（dataclass）中，直接为字段指定可变默认值（如列表或字典 items: list = []）会引发严重的陷阱：所有类实例会共享同一个默认对象。这不仅会被 Python 编译器直接报错，还会引发隐蔽的数据污染问题。掌握 field(default_factory=list) 是编写健壮、类型安全的高级 Python 代码的关键技能。
- 数据污染
在标准 Python 中，函数或类属性的默认参数在定义时（编译时）只被求值一次。如果你在数据类中写下 tags: list = []，Python 会为所有创建的实例分配同一个列表引用。这意味着修改其中一个实例的列表，会无意间污染其他所有实例的数据。为了防止这种危险行为，Python 强制禁止在数据类中直接使用可变字面量作为默认值，并抛出 ValueError。
- 隔离实例
为了给可变数据类型（如 list、dict、set）提供安全的默认值，我们需要使用标准库 dataclasses 中的 field() 函数，并配合 default_factory 参数。default_factory 接受一个无参数的可调用对象（如 list、dict 或自定义函数），在每次实例化对象时被独立调用，从而为每个实例生成一个全新的、完全隔离的独立对象。

In [7]:
@dataclass
class Product:
    product_id: int
    name: str
    price: float
    # item: list=[]  会触发Valueerror

from dataclasses import field, dataclass
from typing import List

@dataclass
class Product:
    product_id: int
    name: str
    price: float
    item: List[str] = field(default_factory=list)

cart1 = Product(product_id=101, name='Laptop', price=999.99)
cart1.item.append('Apple Mouse')
cart2 = Product(product_id=101, name='Laptop', price=999.99)
print(cart1)  # 输出: Product(product_id=101, name='Laptop', price=999.99, item=['Apple Mouse'])
print(cart2)  # 输出: Product(product_id=101, name='Laptop', price=999.99, item=[])


Product(product_id=101, name='Laptop', price=999.99, item=['Apple Mouse'])
Product(product_id=101, name='Laptop', price=999.99, item=[])


# 类型提示与高级类型

## Literal
在编写复杂的 Python 代码时，我们经常需要限制一个函数参数或变量只能接受某些特定的固定值。在过去，这通常依靠运行时抛出异常或在文档中说明来解决。而使用 Literal 类型，你可以在静态类型检查阶段（如使用 MyPy）直接约束变量的具体字面量值，从而大幅提高代码的健壮性、可读性并减少低级拼写错误。

In [9]:
from typing import Literal, Union

# 使用默认超时或永不超时
TimeoutType = Union[Literal['default','infinite'], float, int]
# 或者写成 TimeoutType = Literal['default','infinite'] | float | int]

def connect_to_server(url: str, timeout: TimeoutType) -> None:
    if timeout == 'default':
        print(f"使用系统默认超时时间")
    elif timeout == 'infinite':
        print(f"使用永不超时")
    else:
        print(f"已连接到 {url}，超时时间设置为: {timeout} 秒")

# 正确的调用方式
connect_to_server("https://example.com", 'default')  # 使用系统默认超时时间
connect_to_server("https://example.com", 'infinite')  # 使用永不超时
# 错误的调用方式
# connect_to_server("https://example.com", 'forever')  # 这将引发类型检查错误

使用系统默认超时时间
使用永不超时


## callable
在编写复杂的 Python 代码时，我们经常需要将函数作为参数传递给另一个函数，或者编写高阶函数。Callable 是 typing 模块中的一个重要类型提示工具，它允许我们精确地指定一个对象是否可调用（例如函数、方法或实现了 `__call__` 的类实例），并能约束其参数类型和返回值类型。掌握 Callable 可以让你的代码类型检查更加严密，大幅减少运行时类型错误。

In [10]:
from typing import Callable

def execute_operation(a: int, b: int, operation: Callable[[int, int], int]) -> int:
    return operation(a, b)

def add(x: int, y: int) -> int:
    return x + y

result = execute_operation(5, 3, add)
print(f"Result of addition: {result}")  # 输出: Result of addition: 8

Result of addition: 8
